# Conference Organizer(학회 시간표 앱) 만들기 실습 / Building the Conference Organizer

학회 프로그램 데이터(JSON)를 **파이썬**으로 HTML 템플릿에 넣어서,
인터넷 없이도 내 PC 브라우저에서 열리는 **HTML 파일 하나짜리 웹앱**을 만듭니다.

| 단계 | 하는 일 | 배우는 것 |
|---|---|---|
| 준비 | 도우미 함수 정의, 데이터 불러오기 | 파일 업로드, JSON 읽기 |
| Step 0 | 데이터 살펴보기 | JSON 구조, pandas 표 |
| Step 1 | 파이썬만으로 HTML 만들기 | f-string, `html.escape` |
| Step 2 | JS 로 발표 목록 그리기 | 배열 → HTML, `innerHTML`, 이스케이프 |
| Step 3 | 날짜·장소 필터, 검색 | 상태(state) + `render()` 패턴, 이벤트 |
| Step 4 | ☆ 북마크와 내 일정 | `localStorage`, 시간 겹침 계산 |
| Step 5 | 완성본 | 탭·상세 화면(주소 `#` 라우팅), 북마크 내보내기 |
| 과제 | 직접 기능 추가 | |

모든 단계는 같은 순서로 진행합니다.

```
템플릿 셀(%%writefile) 실행  →  build()  →  preview()  →  download()  →  내 PC 에서 열기
```

> ⚠️ **파일 → 드라이브에 사본 저장** 을 먼저 하고, 사본에서 작업하세요.
> 셀은 위에서부터 순서대로 실행합니다 (`Ctrl + Enter` 또는 ▶).
> 브라우저는 **PC 의 Chrome** 을 권장합니다.

---

🇬🇧 **English**

You put conference program data (JSON) into an HTML template with **Python** and build a **single-file HTML web app**
that opens in your PC browser even without internet.

| Stage | What you do | What you learn |
|---|---|---|
| Setup | define helper functions, load the data | file upload, reading JSON |
| Step 0 | explore the data | JSON structure, pandas tables |
| Step 1 | make HTML with Python only | f-strings, `html.escape` |
| Step 2 | draw the talk list with JS | array → HTML, `innerHTML`, escaping |
| Step 3 | date/room filters, search | state + `render()` pattern, events |
| Step 4 | ☆ bookmarks and My Schedule | `localStorage`, finding time clashes |
| Step 5 | finished app | tabs, detail view (`#` hash routing), bookmark export |
| Assignment | add your own features | |

Every step follows the same order:

```
run the template cell (%%writefile)  →  build()  →  preview()  →  download()  →  open it on your PC
```

> ⚠️ First do **File → Save a copy in Drive** and work in the copy.
> Run the cells in order from the top (`Ctrl + Enter` or ▶). **Chrome on a PC** is recommended.

## 준비 1. 도우미 함수 / Setup 1. Helper functions

아래 셀을 실행하면 함수 세 개가 만들어집니다. 내용은 몰라도 되지만, 이름과 하는 일은 알아 두세요.

| 함수 | 하는 일 |
|---|---|
| `build("템플릿.html", "결과.html")` | 템플릿의 `__DATA__` 자리에 데이터를 넣어 결과 HTML 을 만든다 |
| `preview("결과.html")` | Colab 화면 안에서 결과 HTML 을 띄워 본다 |
| `download("결과.html")` | 결과 HTML 을 내 PC 로 내려받는다 |

---

🇬🇧 **English**

Running the cell below creates three functions. You don't need to understand their code, but remember their names and what they do.

| Function | What it does |
|---|---|
| `build("template.html", "result.html")` | puts the data into the template's `__DATA__` spot and writes the result HTML |
| `preview("result.html")` | shows the result HTML inside Colab |
| `download("result.html")` | downloads the result HTML to your PC |

In [ ]:
import json, os, html, subprocess, sys, time

DATA = None

def load_data(path):
    """conference.json 을 읽어 DATA 에 넣고 개수를 보여 준다.
    Reads conference.json into DATA and prints the counts."""
    global DATA
    with open(path, encoding="utf-8") as f:
        DATA = json.load(f)
    meta = DATA["meta"]
    print(f"{meta['name']} · {meta.get('place', '')} · {meta['days'][0]} ~ {meta['days'][-1]}")
    print(f"sessions {len(DATA['sessions'])}, talks {len(DATA['talks'])}, abstracts {len(DATA['abstracts'])}")

def build(template_path, out_path):
    """템플릿의 __DATA__ 자리에 JSON 데이터를 넣어 HTML 파일 하나를 만든다.
    Puts the JSON data into the template's __DATA__ spot and writes one HTML file."""
    if DATA is None:
        raise RuntimeError("데이터가 없습니다. 위의 '준비 2. 데이터 불러오기' 셀을 먼저 실행하세요. / "
                           "No data loaded. Run the 'Setup 2. Load the data' cell above first.")
    with open(template_path, encoding="utf-8") as f:
        page = f.read()
    n = page.count("__DATA__")
    if n == 0:
        raise ValueError(f"{template_path} 에 __DATA__ 자리가 없습니다. "
                         "템플릿에 const DATA = __DATA__; 줄이 그대로 있는지 확인하세요. / "
                         f"{template_path} has no __DATA__ spot. Check that the line const DATA = __DATA__; is still there.")
    if n > 1:
        raise ValueError(f"{template_path} 에 __DATA__ 가 {n}번 있습니다. 한 번만 있어야 합니다 "
                         "(주석에 써 둔 __DATA__ 도 지우세요). / "
                         f"{template_path} contains __DATA__ {n} times; it must appear exactly once "
                         "(also remove any __DATA__ written in comments).")
    # 데이터 안에 "</script>" 같은 글자가 있으면 <script> 가 거기서 끝나 버린다.
    # 그래서 </ 를 <\/ 로 바꿔 둔다 (JS 에서는 둘 다 같은 문자열이다).
    # EN: If the data contains text like "</script>", the <script> block would end right there,
    # EN: so we turn </ into <\/ (both mean the same string in JS).
    js = json.dumps(DATA, ensure_ascii=False).replace("</", "<\\/")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(page.replace("__DATA__", js))
    print(f"{out_path} 완성 / done ({os.path.getsize(out_path) / 1e6:.1f} MB)")

PORT = 8000
_server = None

def preview(path, height=650):
    """Colab 안에 작은 웹 서버를 띄우고 그 화면을 아래에 보여 준다.
    Starts a small web server inside Colab and shows the page below the cell."""
    global _server
    try:
        from google.colab import output
    except ImportError:
        print("Colab 이 아니면 이 파일을 브라우저로 직접 여세요 / Not in Colab — open this file in your browser:", os.path.abspath(path))
        return
    if _server is None:
        _server = subprocess.Popen([sys.executable, "-m", "http.server", str(PORT)],
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(1)
    # 뒤에 ?v=시각 을 붙여 브라우저가 예전 파일을 다시 쓰지 않게 한다
    # EN: add ?v=<time> so the browser doesn't reuse an old cached file
    output.serve_kernel_port_as_iframe(PORT, path=f"/{path}?v={int(time.time())}", height=height)

def download(path):
    """내 PC 로 내려받기. 반응이 없으면 왼쪽 📁 파일 패널에서 파일 → ⋮ → 다운로드.
    Download to your PC. If nothing happens, use the 📁 Files panel on the left → the file → ⋮ → Download."""
    try:
        from google.colab import files
        files.download(path)
    except ImportError:
        print("Colab 이 아닙니다. 파일 위치 / Not in Colab. File location:", os.path.abspath(path))

print("준비 완료 / ready: build, preview, download")

## 준비 2. 데이터 불러오기 / Setup 2. Load the data

강사가 안내한 방법 **하나만** 실행하세요.
`sessions …, talks …, abstracts …` 처럼 개수가 출력되면 성공입니다.

> 런타임이 초기화되면(오래 쉬었을 때 등) 업로드한 파일이 사라집니다. 그때는 준비 1 부터 다시 실행하세요.

### 방법 A — 파일 업로드
셀을 실행하면 **파일 선택** 버튼이 나옵니다 → 받아 둔 `conference.json` 을 고르세요.

---

🇬🇧 **English**

Run **only one** of the two methods, whichever your instructor tells you.
It worked if you see counts like `sessions …, talks …, abstracts …`.

> When the runtime is reset (e.g. after a long break), uploaded files disappear. Then run again from Setup 1.

### Method A — upload the file
Running the cell shows a **Choose Files** button → pick the `conference.json` you received.

In [ ]:
from google.colab import files

uploaded = files.upload()            # 파일 선택 버튼이 나타난다 / a "Choose Files" button appears
name = next(iter(uploaded))          # 이름이 달라도(conference (1).json 등) 그대로 쓴다 / works even if renamed (e.g. "conference (1).json")
if name != "conference.json":
    os.replace(name, "conference.json")
load_data("conference.json")

### 방법 B — 구글 드라이브 연결
1. 강사가 공유한 `conference.json` 을 **내 드라이브에 바로가기 추가**(또는 사본)해 둡니다.
2. 셀을 실행하면 권한 창이 뜹니다 → 내 계정 선택 → **허용**.
3. `DATA_PATH` 를 내 파일 위치에 맞게 고칩니다 (왼쪽 📁 파일 패널 → `drive/MyDrive` 에서 파일 → ⋮ → **경로 복사**).

---

🇬🇧 **English**

### Method B — connect Google Drive
1. Add a **shortcut to My Drive** (or a copy) of the `conference.json` your instructor shared.
2. Running the cell opens a permission window → choose your account → **Allow**.
3. Change `DATA_PATH` to where your file is (📁 Files panel on the left → `drive/MyDrive` → the file → ⋮ → **Copy path**).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/conference.json"   # ← 내 파일 위치로 고치기 / change to your file's location
load_data(DATA_PATH)

## Step 0. 데이터 살펴보기 / Explore the data

`DATA` 는 파이썬 **딕셔너리**입니다. 네 부분으로 되어 있어요.

| 키 | 내용 | 주요 필드 |
|---|---|---|
| `meta` | 학회 정보 | `name`(학회명), `place`, `days`(날짜 목록) |
| `sessions` | 세션 | `code`, `title` |
| `talks` | 발표 일정 | `id`, `date`, `day_label`, `time_start`, `time_end`, `room`, `session`, `title`, `first_author`, `kind`(`talk`/`plenary`), `abstract_id` |
| `abstracts` | 초록 | `id`, `session`, `title`, `authors[]`, `affiliations[]`, `abstract`, `keywords[]` |

발표와 초록은 `talks[i].abstract_id` ↔ `abstracts[j].id` 로 연결됩니다.
`abstract_id` 가 비어 있는(`None`) 발표도 있습니다(워크숍 등). 포스터처럼 발표 일정이 없는 초록도 있고,
데이터에 따라 초록 본문(`abstract`)이 비어 있을 수 있습니다. 표로 보면 훨씬 알아보기 쉽습니다.

---

🇬🇧 **English**

`DATA` is a Python **dictionary** with four parts.

| Key | Contents | Main fields |
|---|---|---|
| `meta` | conference info | `name`, `place`, `days` (list of dates) |
| `sessions` | sessions | `code`, `title` |
| `talks` | talk schedule | `id`, `date`, `day_label`, `time_start`, `time_end`, `room`, `session`, `title`, `first_author`, `kind` (`talk`/`plenary`), `abstract_id` |
| `abstracts` | abstracts | `id`, `session`, `title`, `authors[]`, `affiliations[]`, `abstract`, `keywords[]` |

Talks and abstracts are linked by `talks[i].abstract_id` ↔ `abstracts[j].id`.
Some talks have an empty (`None`) `abstract_id` (workshops etc.). Some abstracts have no scheduled talk (posters),
and depending on the data the abstract text (`abstract`) may be empty. A table makes all this much easier to see.

In [ ]:
import pandas as pd

talks = pd.DataFrame(DATA["talks"])
abstracts = pd.DataFrame(DATA["abstracts"])
talks.head()

In [ ]:
# 날짜별·장소별 발표 수
# EN: number of talks per day and per room
pd.crosstab(talks["day_label"], talks["room"], margins=True)

In [ ]:
# 발표 하나와, 그 발표에 연결된 초록 하나를 그대로 출력해 보기
# EN: print one talk and the abstract linked to it, as they are
t = next(t for t in DATA["talks"] if t["abstract_id"])
a = next(a for a in DATA["abstracts"] if a["id"] == t["abstract_id"])
print(json.dumps(t, ensure_ascii=False, indent=2))
print(json.dumps({**a, "abstract": a["abstract"][:200] + "…"}, ensure_ascii=False, indent=2))

**해 볼 것**: 초록이 가장 많은 세션은? 키워드 중 가장 많이 나오는 것은?
(힌트: `abstracts["session"].value_counts()`, `abstracts["keywords"].explode().value_counts()`)

---

🇬🇧 **English**

**Try it**: Which session has the most abstracts? Which keyword appears most often?
(Hint: `abstracts["session"].value_counts()`, `abstracts["keywords"].explode().value_counts()`)

In [ ]:
# 여기에 직접 해 보세요
# EN: try it yourself here

## Step 1. 파이썬만으로 HTML 만들기 / HTML with Python only

웹 페이지는 결국 **글자(텍스트) 파일**입니다. 파이썬 f-string 으로 HTML 글자를 만들어 파일에 쓰면 끝!

⚠️ 제목에 `<` 나 `&` 같은 글자가 있으면 HTML 이 깨집니다. 그래서 `html.escape()` 로 바꿔 넣습니다.

---

🇬🇧 **English**

A web page is in the end just a **text file**. Build the HTML text with Python f-strings, write it to a file — done!

⚠️ Characters like `<` or `&` in a title break the HTML, so we pass them through `html.escape()`.

In [ ]:
# 첫날 발표만 골라 <li> 목록을 글자로 만든다
# EN: pick the first day's talks and build an <li> list as text
first_day = DATA["meta"]["days"][0]
day_talks = [t for t in DATA["talks"] if t["date"] == first_day]

items = []
for t in day_talks:
    items.append(f"""
    <li>
      <b>{t["time_start"]}–{t["time_end"]}</b> · {html.escape(t["room"])}<br>
      {html.escape(t["title"])}<br>
      <small>{html.escape(t["first_author"] or "")}</small>
    </li>""")

page = f"""<!doctype html>
<html lang="ko">
<head><meta charset="utf-8"><title>{html.escape(DATA["meta"]["name"])}</title></head>
<body>
  <h1>{html.escape(DATA["meta"]["name"])} — {html.escape(day_talks[0]["day_label"])}</h1>
  <p>발표 {len(day_talks)}건 / {len(day_talks)} talks</p>
  <ul>{"".join(items)}</ul>
</body>
</html>"""

with open("step1.html", "w", encoding="utf-8") as f:
    f.write(page)
print("step1.html 완성 / done")
preview("step1.html", height=400)

잘 나오죠? 그런데 **날짜를 바꿔 보거나, 검색하려면** 어떻게 해야 할까요?
파이썬은 파일을 만들 때 한 번만 실행되니, 날짜마다 · 검색어마다 파일을 따로 만들어야 합니다. 😵

→ 그래서 **데이터는 통째로 HTML 에 넣어 두고**, 화면은 브라우저 안의 **자바스크립트(JS)** 가 그리게 합니다.

**해 볼 것**: 둘째 날 목록으로 바꿔 보기, 기조강연(`kind == "plenary"`)만 굵게 표시하기.

---

🇬🇧 **English**

Looks good? But how would you **switch the date or search**?
Python runs only once when it makes the file, so you'd need a separate file for every date and every search term. 😵

→ So we **put all the data into the HTML** and let **JavaScript (JS)** in the browser draw the screen.

**Try it**: switch to the second day's list; make only plenary talks (`kind == "plenary"`) bold.

In [ ]:
download("step1.html")      # 내려받아 더블클릭으로 열어 보기 / download it and open it by double-clicking

## Step 2. JS 로 목록 그리기 / Draw the list with JS

이제부터는 **템플릿 HTML** 을 쓰고, 파이썬은 데이터만 넣어 줍니다.

```html
<script>
const DATA = __DATA__;      ← build() 가 이 자리를 JSON 으로 바꾼다
</script>
```

JSON 은 원래 자바스크립트 문법에서 나온 것이라, 그대로 넣으면 JS 객체가 됩니다.
그다음은 JS 가 `DATA.talks.filter(...)` → `` `<div>${...}</div>` `` (템플릿 리터럴) → `innerHTML` 순서로 화면을 그립니다.
JS 에서도 `esc()` 로 이스케이프하는 것을 잊지 마세요.

아래 셀은 `%%writefile` 로 템플릿 파일을 만듭니다. **셀 안의 HTML 을 고친 뒤 다시 실행**하면 바뀐 템플릿이 저장됩니다.

> **템플릿 셀 규칙** — 첫 줄 `%%writefile …` 과 `const DATA = __DATA__;` 의 `__DATA__` 는 그대로 두세요.
> 템플릿을 고쳤다면 **템플릿 셀 → 아래 build 셀** 순서로 둘 다 다시 실행해야 반영됩니다.
> 화면이 하얗게 나오면 내려받은 파일을 Chrome 으로 열고 `F12` → **Console** 에서 빨간 오류를 확인하세요.

---

🇬🇧 **English**

From here on we use a **template HTML**, and Python only inserts the data.

```html
<script>
const DATA = __DATA__;      ← build() replaces this spot with the JSON
</script>
```

JSON comes from JavaScript syntax, so pasted in as-is it becomes a JS object.
Then JS draws the screen: `DATA.talks.filter(...)` → `` `<div>${...}</div>` `` (template literal) → `innerHTML`.
Don't forget to escape with `esc()` in JS too.

The cell below creates the template file with `%%writefile`. **Edit the HTML in the cell and run it again** to save the changed template.

> **Template cell rules** — leave the first line `%%writefile …` and the `__DATA__` in `const DATA = __DATA__;` as they are.
> After editing a template, re-run **the template cell, then the build cell below it** for the change to take effect.
> If the page comes out blank white, open the downloaded file in Chrome and look for red errors in `F12` → **Console**.

In [ ]:
%%writefile step2_list.html
<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Step 2 · 발표 목록</title>
<style>
  body { font-family: system-ui, "Noto Sans KR", sans-serif; margin: 0; background: #f4f5f7; color: #1d2733; }
  header { background: #0d3b66; color: #fff; padding: 12px 16px; }
  header h1 { margin: 0; font-size: 18px; }
  main { max-width: 720px; margin: 0 auto; padding: 12px; }
  .talk { background: #fff; border: 1px solid #e4e7ec; border-radius: 10px; padding: 10px 12px; margin-bottom: 8px; }
  .time { font-weight: 700; color: #0d3b66; font-size: 13px; }
  .room { color: #6b7686; font-size: 12px; margin-left: 6px; }
  .title { margin: 4px 0 2px; font-weight: 600; }
  .author { color: #6b7686; font-size: 13px; }
</style>
</head>
<body>
<header><h1 id="conf-name"></h1></header>
<main id="list"></main>

<script>
// 파이썬이 이 자리에 JSON 데이터를 넣어 준다.
// EN: Python puts the JSON data in this spot.
const DATA = __DATA__;

// [Step 2] 데이터 배열 → HTML 문자열 → 화면
// EN: [Step 2] data array → HTML string → screen
document.getElementById("conf-name").textContent = DATA.meta.name;

const firstDay = DATA.meta.days[0];
const talks = DATA.talks.filter(t => t.date === firstDay);

// 사용자 데이터를 innerHTML 에 넣을 때는 꼭 이스케이프한다 (<, & 같은 문자 대비)
// EN: Always escape data before putting it into innerHTML (in case of characters like <, &)
function esc(s) {
  return String(s ?? "").replace(/[&<>"]/g, c => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;" }[c]));
}

document.getElementById("list").innerHTML = talks.map(t => `
  <div class="talk">
    <span class="time">${t.time_start}–${t.time_end}</span>
    <span class="room">${esc(t.room)}</span>
    <div class="title">${esc(t.title)}</div>
    <div class="author">${esc(t.first_author)}</div>
  </div>`).join("");

// 해 볼 것: 발표 수를 제목 옆에 표시해 보기, 세션 코드(t.session)도 보여 주기
// EN: Try: show the number of talks next to the title; also show the session code (t.session)
</script>
</body>
</html>

In [ ]:
build("step2_list.html", "out_step2.html")
preview("out_step2.html")

In [ ]:
download("out_step2.html")

## Step 3. 날짜·장소 필터와 검색 / Date & room filters, search

화면에 영향을 주는 값(고른 날짜, 고른 장소, 검색어)을 `state` 한곳에 모아 둡니다.

1. 사용자가 칩을 누르거나 글자를 입력하면 → **`state` 만 바꾸고**
2. **`render()`** 가 `state` 를 보고 화면 전체를 다시 그립니다.

이 **state → render** 패턴은 React 같은 큰 프레임워크도 똑같이 씁니다.
장소 칩 목록은 데이터에서 뽑아내므로(`new Set(DATA.talks.map(t => t.room))`) 다른 학회 데이터를 넣어도 그대로 동작합니다.

---

🇬🇧 **English**

Keep every value that affects the screen (selected date, selected room, search term) together in one `state`.

1. When the user clicks a chip or types → **only change `state`**
2. **`render()`** looks at `state` and redraws the whole screen.

Big frameworks like React use exactly this **state → render** pattern.
The room chips are taken from the data (`new Set(DATA.talks.map(t => t.room))`), so it still works with another conference's data.

In [ ]:
%%writefile step3_filter.html
<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Step 3 · 날짜·장소 필터와 검색</title>
<style>
  body { font-family: system-ui, "Noto Sans KR", sans-serif; margin: 0; background: #f4f5f7; color: #1d2733; }
  header { background: #0d3b66; color: #fff; padding: 12px 16px; position: sticky; top: 0; }
  header h1 { margin: 0; font-size: 18px; }
  main { max-width: 720px; margin: 0 auto; padding: 12px; }
  .chips { display: flex; gap: 6px; overflow-x: auto; padding-bottom: 8px; }
  .chip { flex: none; border: 1px solid #e4e7ec; background: #fff; border-radius: 999px; padding: 6px 12px; font-size: 13px; cursor: pointer; }
  .chip.on { background: #0d3b66; color: #fff; border-color: #0d3b66; }
  input[type=search] { width: 100%; box-sizing: border-box; padding: 9px 12px; font-size: 15px; border: 1px solid #ccd; border-radius: 8px; margin-bottom: 10px; }
  .count { color: #6b7686; font-size: 13px; margin: 0 2px 8px; }
  .talk { background: #fff; border: 1px solid #e4e7ec; border-radius: 10px; padding: 10px 12px; margin-bottom: 8px; }
  .time { font-weight: 700; color: #0d3b66; font-size: 13px; }
  .room { color: #6b7686; font-size: 12px; margin-left: 6px; }
  .title { margin: 4px 0 2px; font-weight: 600; }
  .author { color: #6b7686; font-size: 13px; }
  mark { background: #ffe58a; }
</style>
</head>
<body>
<header><h1 id="conf-name"></h1></header>
<main>
  <div class="chips" id="days"></div>
  <div class="chips" id="rooms"></div>
  <input type="search" id="q" placeholder="제목·저자 검색">
  <div class="count" id="count"></div>
  <div id="list"></div>
</main>

<script>
const DATA = __DATA__;

function esc(s) {
  return String(s ?? "").replace(/[&<>"]/g, c => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;" }[c]));
}

// [Step 3] 화면 상태(state)를 한곳에 두고, 바뀔 때마다 render() 로 다시 그린다
// EN: [Step 3] keep the screen state in one place and redraw everything with render() whenever it changes
const state = { day: DATA.meta.days[0], room: "전체", q: "" };
const rooms = ["전체", ...new Set(DATA.talks.map(t => t.room))].sort((a, b) => a === "전체" ? -1 : b === "전체" ? 1 : a.localeCompare(b));

function chipsHTML(values, current, label = v => v) {
  return values.map(v => `<button class="chip ${v === current ? "on" : ""}" data-v="${esc(v)}">${esc(label(v))}</button>`).join("");
}

function highlight(text, q) {
  const safe = esc(text);
  if (!q) return safe;
  const re = new RegExp(q.replace(/[.*+?^${}()|[\]\\]/g, "\\$&"), "gi");
  return safe.replace(re, m => `<mark>${m}</mark>`);
}

function render() {
  document.getElementById("days").innerHTML = chipsHTML(DATA.meta.days, state.day, d => DATA.talks.find(t => t.date === d).day_label);
  document.getElementById("rooms").innerHTML = chipsHTML(rooms, state.room);

  const q = state.q.trim().toLowerCase();
  const talks = DATA.talks.filter(t =>
    // 검색어가 있으면 날짜와 상관없이 전체에서 찾는다
    // EN: if there is a search term, search all days regardless of the selected date
    (q ? true : t.date === state.day) &&
    (state.room === "전체" || t.room === state.room) &&
    (!q || t.title.toLowerCase().includes(q) || (t.first_author || "").toLowerCase().includes(q))
  );

  document.getElementById("count").textContent = `${talks.length}건`;
  document.getElementById("list").innerHTML = talks.map(t => `
    <div class="talk">
      <span class="time">${q ? t.day_label + " · " : ""}${t.time_start}–${t.time_end}</span>
      <span class="room">${esc(t.room)}${t.session ? " · " + t.session : ""}</span>
      <div class="title">${highlight(t.title, q)}</div>
      <div class="author">${highlight(t.first_author, q)}</div>
    </div>`).join("");
}

// 이벤트: 칩을 누르거나 검색어를 입력하면 state 를 바꾸고 다시 그린다
// EN: Events: clicking a chip or typing a search term changes state, then redraws
document.getElementById("days").addEventListener("click", e => {
  if (e.target.dataset.v) { state.day = e.target.dataset.v; render(); }
});
document.getElementById("rooms").addEventListener("click", e => {
  if (e.target.dataset.v) { state.room = e.target.dataset.v; render(); }
});
document.getElementById("q").addEventListener("input", e => { state.q = e.target.value; render(); });

document.getElementById("conf-name").textContent = DATA.meta.name;
render();

// 해 볼 것: 세션 코드로도 거르는 칩 줄을 추가해 보기
// EN: Try: add another row of chips that filters by session code
</script>
</body>
</html>

In [ ]:
build("step3_filter.html", "out_step3.html")
preview("out_step3.html")

In [ ]:
download("out_step3.html")

## Step 4. 북마크와 내 일정 / Bookmarks and My Schedule

☆ 를 누르면 발표 id 를 `Set` 에 넣고 **`localStorage`** 에 저장합니다.
`localStorage` 는 브라우저가 사이트(파일)별로 주는 작은 저장 공간이라, 페이지를 닫았다 열어도 북마크가 남아 있어요.

- 저장은 **글자만** 됩니다 → `JSON.stringify` 로 바꿔 저장, `JSON.parse` 로 꺼내기
- 브라우저 설정에 따라 막힐 수 있으므로 `try / catch` 로 감쌉니다
- **내 일정** 탭: 날짜별로 모아 시간순으로 보여 주고, 시간이 겹치는 발표는 빨갛게 표시합니다.
  `"09:30" < "10:00"` 처럼 `HH:MM` 글자는 그대로 비교해도 시간 순서가 맞습니다.
- 목록은 계속 다시 그려지므로, 클릭 이벤트는 바깥(`main`)에 **한 번만** 걸고 무엇을 눌렀는지 확인합니다 (이벤트 위임).

⚠️ Colab 미리보기와 내려받은 파일은 **서로 다른 저장 공간**을 씁니다. 미리보기에서 찍은 북마크는 PC 파일에 없어요.

---

🇬🇧 **English**

Clicking ☆ puts the talk id into a `Set` and saves it to **`localStorage`**.
`localStorage` is a small storage space the browser gives each site (file), so bookmarks survive closing and reopening the page.

- It stores **text only** → save with `JSON.stringify`, read back with `JSON.parse`
- It can be blocked by browser settings, so wrap it in `try / catch`
- **My Schedule** tab: groups bookmarks by date in time order and marks clashing talks in red.
  `HH:MM` strings compare correctly as text, e.g. `"09:30" < "10:00"`.
- The list keeps being redrawn, so attach the click listener **once** to the outer element (`main`) and check what was clicked (event delegation).

⚠️ The Colab preview and the downloaded file use **separate storage**. Bookmarks made in the preview are not in the file on your PC.

In [ ]:
%%writefile step4_bookmark.html
<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Step 4 · 북마크와 내 일정</title>
<style>
  body { font-family: system-ui, "Noto Sans KR", sans-serif; margin: 0; background: #f4f5f7; color: #1d2733; }
  header { background: #0d3b66; color: #fff; padding: 12px 16px; position: sticky; top: 0; }
  header h1 { margin: 0; font-size: 18px; }
  main { max-width: 720px; margin: 0 auto; padding: 12px; }
  .chips { display: flex; gap: 6px; overflow-x: auto; padding-bottom: 8px; }
  .chip { flex: none; border: 1px solid #e4e7ec; background: #fff; border-radius: 999px; padding: 6px 12px; font-size: 13px; cursor: pointer; }
  .chip.on { background: #0d3b66; color: #fff; border-color: #0d3b66; }
  input[type=search] { width: 100%; box-sizing: border-box; padding: 9px 12px; font-size: 15px; border: 1px solid #ccd; border-radius: 8px; margin-bottom: 10px; }
  .count { color: #6b7686; font-size: 13px; margin: 0 2px 8px; }
  .talk { background: #fff; border: 1px solid #e4e7ec; border-radius: 10px; padding: 10px 12px; margin-bottom: 8px; }
  .time { font-weight: 700; color: #0d3b66; font-size: 13px; }
  .room { color: #6b7686; font-size: 12px; margin-left: 6px; }
  .title { margin: 4px 0 2px; font-weight: 600; }
  .author { color: #6b7686; font-size: 13px; }
  mark { background: #ffe58a; }
  /* [Step 4] 탭, 북마크 버튼, 내 일정 / tabs, bookmark button, My Schedule */
  .tabs { display: flex; gap: 6px; margin-bottom: 10px; }
  .tab { flex: 1; border: 1px solid #e4e7ec; background: #fff; border-radius: 8px; padding: 8px; font-size: 14px; font-weight: 600; cursor: pointer; color: #6b7686; }
  .tab.on { background: #0d3b66; color: #fff; border-color: #0d3b66; }
  .talk { display: flex; gap: 8px; align-items: flex-start; }
  .talk-body { flex: 1; min-width: 0; }
  .bm { flex: none; background: none; border: none; font-size: 22px; line-height: 1; color: #6b7686; cursor: pointer; padding: 2px; }
  .bm.on { color: #ee9b00; }
  .day-h { font-size: 14px; color: #0d3b66; margin: 16px 2px 8px; }
  .talk.clash { border-color: #e23b3b; background: #fff5f5; }
  .clash-badge { color: #e23b3b; font-size: 12px; font-weight: 700; margin-left: 6px; }
  .empty { color: #6b7686; padding: 24px 4px; text-align: center; }
</style>
</head>
<body>
<header><h1 id="conf-name"></h1></header>
<main>
  <div class="tabs" id="tabs"></div>
  <div id="program-view">
    <div class="chips" id="days"></div>
    <div class="chips" id="rooms"></div>
    <input type="search" id="q" placeholder="제목·저자 검색">
    <div class="count" id="count"></div>
    <div id="list"></div>
  </div>
  <div id="my-view"></div>
</main>

<script>
const DATA = __DATA__;

function esc(s) {
  return String(s ?? "").replace(/[&<>"]/g, c => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;" }[c]));
}

// [Step 4] 북마크: 발표 id 들을 Set 에 담고, 바뀔 때마다 localStorage 에 저장한다.
// localStorage 는 브라우저 설정(시크릿 창, 쿠키 차단 등)에 따라 막힐 수 있으므로 try/catch 로 감싼다.
// EN: [Step 4] Bookmarks: keep talk ids in a Set and save it to localStorage on every change.
// EN: localStorage can be blocked by browser settings (incognito, blocked cookies …), so wrap it in try/catch.
const BM_KEY = "gsk2026.bookmarks";
function loadBookmarks() {
  try { return new Set(JSON.parse(localStorage.getItem(BM_KEY) || "[]")); }
  catch (e) { return new Set(); }
}
function saveBookmarks() {
  try { localStorage.setItem(BM_KEY, JSON.stringify([...bookmarks])); }
  catch (e) { /* 저장이 막혀도 이번 화면에서는 계속 쓸 수 있다 / even if saving is blocked, it still works on this page */ }
}
const bookmarks = loadBookmarks();

// [Step 3] 화면 상태(state)를 한곳에 두고, 바뀔 때마다 render() 로 다시 그린다
// EN: [Step 3] keep the screen state in one place and redraw everything with render() whenever it changes
// [Step 4] tab: "program"(전체 프로그램) 또는 "my"(내 일정)
// EN: [Step 4] tab: "program" (full program) or "my" (My Schedule)
const state = { tab: "program", day: DATA.meta.days[0], room: "전체", q: "" };
const rooms = ["전체", ...new Set(DATA.talks.map(t => t.room))].sort((a, b) => a === "전체" ? -1 : b === "전체" ? 1 : a.localeCompare(b));

function chipsHTML(values, current, label = v => v) {
  return values.map(v => `<button class="chip ${v === current ? "on" : ""}" data-v="${esc(v)}">${esc(label(v))}</button>`).join("");
}

function highlight(text, q) {
  const safe = esc(text);
  if (!q) return safe;
  const re = new RegExp(q.replace(/[.*+?^${}()|[\]\\]/g, "\\$&"), "gi");
  return safe.replace(re, m => `<mark>${m}</mark>`);
}

// [Step 4] 발표 카드 하나를 그리는 함수 — 프로그램 목록과 내 일정에서 같이 쓴다
// EN: [Step 4] draws one talk card — shared by the program list and My Schedule
function talkHTML(t, { q = "", showDay = false, clash = false } = {}) {
  const on = bookmarks.has(t.id);
  return `
    <div class="talk ${clash ? "clash" : ""}">
      <div class="talk-body">
        <span class="time">${showDay ? t.day_label + " · " : ""}${t.time_start}–${t.time_end}</span>
        <span class="room">${esc(t.room)}${t.session ? " · " + esc(t.session) : ""}</span>
        ${clash ? '<span class="clash-badge">⚠ 시간 겹침</span>' : ""}
        <div class="title">${highlight(t.title, q)}</div>
        <div class="author">${highlight(t.first_author, q)}</div>
      </div>
      <button class="bm ${on ? "on" : ""}" data-id="${t.id}" title="북마크">${on ? "★" : "☆"}</button>
    </div>`;
}

// [Step 4] 시간 겹침 찾기: 같은 날 발표를 시작 시각 순으로 놓고,
// 앞 발표들 중 가장 늦게 끝나는 시각보다 내 시작이 이르면 겹친다.
// ("10:30" 같은 문자열은 길이가 같으므로 문자열 비교로도 시간 순서가 맞다)
// EN: [Step 4] Finding time clashes: line up the day's talks by start time;
// EN: a talk clashes if it starts before the latest end time of the talks before it.
// EN: (strings like "10:30" all have the same length, so comparing them as strings gives the right time order)
function findClashes(talks) {
  const clash = new Set();
  let latestEnd = "", latestId = null;
  for (const t of talks) {
    if (t.time_start < latestEnd) { clash.add(t.id); clash.add(latestId); }
    if (t.time_end > latestEnd) { latestEnd = t.time_end; latestId = t.id; }
  }
  return clash;
}

function renderMy() {
  const mine = DATA.talks.filter(t => bookmarks.has(t.id));   // DATA.talks 는 이미 날짜·시간순 / already sorted by date and time
  if (mine.length === 0) {
    return '<div class="empty">아직 북마크한 발표가 없습니다.<br>프로그램에서 ☆ 를 눌러 보세요.</div>';
  }
  let html = "";
  for (const day of DATA.meta.days) {
    const talks = mine.filter(t => t.date === day);
    if (talks.length === 0) continue;
    const clash = findClashes(talks);
    html += `<h2 class="day-h">${esc(talks[0].day_label)} · ${talks.length}건</h2>`;
    html += talks.map(t => talkHTML(t, { clash: clash.has(t.id) })).join("");
  }
  return html;
}

function render() {
  // [Step 4] 탭 버튼과 화면 전환
  // EN: [Step 4] tab buttons and switching views
  document.getElementById("tabs").innerHTML = `
    <button class="tab ${state.tab === "program" ? "on" : ""}" data-tab="program">전체 프로그램</button>
    <button class="tab ${state.tab === "my" ? "on" : ""}" data-tab="my">내 일정 (${bookmarks.size})</button>`;
  document.getElementById("program-view").hidden = state.tab !== "program";
  document.getElementById("my-view").hidden = state.tab !== "my";
  if (state.tab === "my") {
    document.getElementById("my-view").innerHTML = renderMy();
    return;
  }

  document.getElementById("days").innerHTML = chipsHTML(DATA.meta.days, state.day, d => DATA.talks.find(t => t.date === d).day_label);
  document.getElementById("rooms").innerHTML = chipsHTML(rooms, state.room);

  const q = state.q.trim().toLowerCase();
  const talks = DATA.talks.filter(t =>
    // 검색어가 있으면 날짜와 상관없이 전체에서 찾는다
    // EN: if there is a search term, search all days regardless of the selected date
    (q ? true : t.date === state.day) &&
    (state.room === "전체" || t.room === state.room) &&
    (!q || t.title.toLowerCase().includes(q) || (t.first_author || "").toLowerCase().includes(q))
  );

  document.getElementById("count").textContent = `${talks.length}건`;
  document.getElementById("list").innerHTML = talks.map(t => talkHTML(t, { q, showDay: !!q })).join("");
}

// 이벤트: 칩을 누르거나 검색어를 입력하면 state 를 바꾸고 다시 그린다
// EN: Events: clicking a chip or typing a search term changes state, then redraws
document.getElementById("days").addEventListener("click", e => {
  if (e.target.dataset.v) { state.day = e.target.dataset.v; render(); }
});
document.getElementById("rooms").addEventListener("click", e => {
  if (e.target.dataset.v) { state.room = e.target.dataset.v; render(); }
});
document.getElementById("q").addEventListener("input", e => { state.q = e.target.value; render(); });

// [Step 4] 탭 전환
// EN: [Step 4] switching tabs
document.getElementById("tabs").addEventListener("click", e => {
  if (e.target.dataset.tab) { state.tab = e.target.dataset.tab; render(); }
});
// [Step 4] ☆ 버튼: 목록이 다시 그려져도 동작하도록 main 에 한 번만 달아 둔다 (이벤트 위임)
// EN: [Step 4] ☆ button: attach the listener once to <main> so it keeps working after the list is redrawn (event delegation)
document.querySelector("main").addEventListener("click", e => {
  const btn = e.target.closest(".bm");
  if (!btn) return;
  const id = Number(btn.dataset.id);
  if (bookmarks.has(id)) bookmarks.delete(id); else bookmarks.add(id);
  saveBookmarks();
  render();
});

document.getElementById("conf-name").textContent = DATA.meta.name;
render();

// 해 볼 것: 내 일정에서 겹치는 발표 중 하나를 고르면 나머지를 흐리게 표시해 보기,
//          북마크 수가 많은 날짜 칩에 ★ 표시 달아 보기
// EN: Try: when you pick one of the clashing talks in My Schedule, dim the others;
// EN:      put a ★ on date chips that have many bookmarks
</script>
</body>
</html>

In [ ]:
build("step4_bookmark.html", "out_step4.html")
preview("out_step4.html")

In [ ]:
download("out_step4.html")

## Step 5. 완성본 / The finished app

지금까지 배운 것을 모두 합치고 몇 가지를 더했습니다.

- **하단 탭**: 프로그램 / 세션 / 검색 / 내 일정
- **hash 라우팅**: 주소 끝의 `#/talk/12`, `#/session/...` 을 보고 어떤 화면을 그릴지 정합니다.
  파일로 열어도(`file://`) 뒤로가기·새로고침이 됩니다.
- **발표 상세**: 초록 본문, 저자, 소속, 키워드 (키워드를 누르면 검색)
- **검색**: 제목·저자·키워드에 더해 **초록 본문**까지. 본문에서만 찾으면 앞뒤 문장을 보여 줍니다.
- **`Map` 색인**: `id → 발표` 를 미리 만들어 두어 빠르게 찾습니다.
- **내보내기/불러오기**: 북마크를 `bookmarks.json` 파일로 저장하고, 다른 PC 에서 불러올 수 있습니다.
- Step 4 와 같은 저장 키를 씁니다. Chrome 은 내 PC 의 HTML 파일들이 저장 공간을 함께 쓰므로 Step 4 에서 한 북마크가 여기서도 보입니다 (정상).

템플릿이 길어서 셀을 접어 두는 게 편합니다. 다 만들었으면 `download()` 로 내려받아 PC 에서 열어 보세요.

---

🇬🇧 **English**

This combines everything so far and adds a few things.

- **Bottom tabs**: Program / Sessions / Search / My Schedule
- **Hash routing**: the end of the address (`#/talk/12`, `#/session/...`) decides which view to draw.
  Back and refresh work even when opened as a file (`file://`).
- **Talk detail**: abstract text, authors, affiliations, keywords (click a keyword to search)
- **Search**: title, authors, keywords and also the **abstract text**. If it matches only the text, the surrounding sentence is shown.
- **`Map` index**: `id → talk` lookups prepared in advance for speed.
- **Export/Import**: save bookmarks to a `bookmarks.json` file and load them on another PC.
- Uses the same storage key as Step 4. Chrome lets local HTML files on your PC share storage, so Step 4 bookmarks show up here too (this is normal).

The template is long, so it's handy to collapse the cell. When you're done, `download()` it and open it on your PC.

In [ ]:
%%writefile app.html
<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<meta name="theme-color" content="#0d3b66">
<title>학회 프로그램</title>
<style>
  :root {
    --navy: #0d3b66; --accent: #ee9b00; --red: #e23b3b;
    --bg: #f4f5f7; --card: #fff; --ink: #1d2733; --muted: #6b7686; --line: #e4e7ec;
    --tabh: 58px;
  }
  * { box-sizing: border-box; -webkit-tap-highlight-color: transparent; }
  html, body { margin: 0; }
  body {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Noto Sans KR", sans-serif;
    background: var(--bg); color: var(--ink); line-height: 1.45;
    padding-bottom: calc(var(--tabh) + env(safe-area-inset-bottom));
  }
  a { color: inherit; text-decoration: none; }
  button { font: inherit; }

  /* 상단 바 / top bar */
  .appbar { position: sticky; top: 0; z-index: 20; background: var(--navy); color: #fff;
    padding: calc(9px + env(safe-area-inset-top)) 16px 9px; }
  .brand-main { font-weight: 800; font-size: 16px; }
  .brand-sub { font-size: 11.5px; color: #cfe0f0; margin-top: 2px; }

  .content { max-width: 720px; margin: 0 auto; padding: 12px 12px 24px; }
  .page-h { font-size: 20px; margin: 8px 4px 12px; }
  .empty, .hint { color: var(--muted); padding: 16px 4px; }
  .hint { padding: 0 4px 10px; font-size: 13px; }

  /* 날짜·장소 칩 / date & room chips */
  .chips { display: flex; gap: 6px; overflow-x: auto; padding: 2px 2px 10px; -webkit-overflow-scrolling: touch; }
  .chip { flex: 0 0 auto; background: var(--card); border: 1px solid var(--line); border-radius: 999px;
    padding: 6px 13px; font-size: 13px; font-weight: 600; color: var(--muted); cursor: pointer; white-space: nowrap; }
  .chip.on { background: var(--navy); border-color: var(--navy); color: #fff; }
  .chips.rooms .chip.on { background: var(--accent); border-color: var(--accent); }

  /* 발표 카드 / talk card */
  .talk { display: flex; gap: 10px; align-items: flex-start; background: var(--card);
    border: 1px solid var(--line); border-radius: 12px; padding: 10px 12px; margin-bottom: 8px; }
  .talk-time { flex: 0 0 auto; width: 44px; font-variant-numeric: tabular-nums; font-weight: 700;
    font-size: 12px; color: var(--navy); line-height: 1.25; padding-top: 2px; }
  .talk-body { flex: 1 1 auto; min-width: 0; }
  .talk-title { display: block; font-size: 14.5px; font-weight: 600; }
  .talk-meta { margin-top: 4px; display: flex; flex-wrap: wrap; gap: 6px; align-items: center; }
  .author, .place { color: var(--muted); font-size: 12.5px; }
  .snippet { margin-top: 5px; color: var(--muted); font-size: 12.5px; }
  .talk.clash { border-color: var(--red); background: #fff5f5; }
  .clash-badge { color: var(--red); font-size: 12px; font-weight: 700; }
  .badge { background: var(--navy); color: #fff; font-size: 11px; font-weight: 700; border-radius: 6px; padding: 1px 6px; }
  .badge.plenary { background: var(--accent); }
  .badge.gray { background: #9aa4b2; }
  .bm { flex: 0 0 auto; background: none; border: none; font-size: 22px; line-height: 1;
    color: var(--muted); cursor: pointer; padding: 2px; }
  .bm.on { color: var(--accent); }
  .bm.big { font-size: 26px; margin-left: auto; }
  .day-h { font-size: 14px; color: var(--navy); margin: 18px 4px 8px; }
  mark { background: #ffe58a; color: inherit; border-radius: 2px; }

  /* 발표 상세 / talk detail */
  .back { background: none; border: none; color: var(--navy); font-weight: 600; padding: 4px 2px 8px; cursor: pointer; }
  .detail-when { display: flex; align-items: center; flex-wrap: wrap; gap: 8px; font-size: 13px; color: var(--muted); margin-bottom: 8px; }
  .detail-title { font-size: 20px; line-height: 1.3; margin: 4px 0 10px; }
  .detail-session { display: flex; gap: 8px; align-items: center; background: var(--card);
    border: 1px solid var(--line); border-radius: 12px; padding: 9px 12px; margin: 0 0 12px;
    font-size: 13.5px; color: var(--navy); }
  .detail-authors { font-weight: 600; margin: 0 0 12px; }
  .detail-authors sup { color: var(--muted); font-weight: 400; }
  .detail-abstract { background: var(--card); border: 1px solid var(--line); border-radius: 12px;
    padding: 14px; font-size: 14.5px; text-align: justify; white-space: pre-line; }
  .kw { display: flex; flex-wrap: wrap; gap: 6px; margin: 12px 0; }
  .kwchip { background: #eef2f7; color: var(--navy); border-radius: 14px; padding: 4px 10px; font-size: 12.5px; cursor: pointer; border: none; }
  .affil { margin-top: 8px; color: var(--muted); font-size: 12.5px; }
  .affil summary { cursor: pointer; }
  .affil ol { padding-left: 18px; }

  /* 세션 목록 / session list */
  .slist { list-style: none; margin: 0; padding: 0; }
  .slist a { display: flex; align-items: center; gap: 10px; background: var(--card);
    border: 1px solid var(--line); border-radius: 12px; padding: 12px; margin-bottom: 8px; }
  .stitle { flex: 1 1 auto; font-size: 14px; font-weight: 600; }
  .scount { color: var(--muted); font-size: 12px; white-space: nowrap; }
  .session-title { font-size: 16px; font-weight: 600; margin: 0 4px 12px; }

  /* 검색 / search */
  .searchbar input { width: 100%; padding: 11px 14px; border: 1px solid var(--line); border-radius: 12px;
    font-size: 16px; background: var(--card); color: var(--ink); margin: 4px 0 10px; }
  .count { color: var(--muted); font-size: 13px; margin: 0 4px 10px; }

  /* 내 일정: 내보내기/불러오기 / My Schedule: export/import */
  .tools { margin: 24px 2px 8px; padding-top: 16px; border-top: 1px solid var(--line); }
  .tools-row { display: flex; gap: 8px; flex-wrap: wrap; }
  .btn { background: var(--card); border: 1px solid var(--line); border-radius: 999px; padding: 8px 14px;
    font-size: 13px; font-weight: 600; color: var(--navy); cursor: pointer; }
  .btn.danger { color: #b03030; border-color: #e3b6b6; }
  .msg { margin-top: 10px; font-size: 13px; color: #1f7a3d; }

  /* 하단 탭 / bottom tabs */
  .tabbar { position: fixed; bottom: 0; left: 0; right: 0; z-index: 20; display: flex;
    background: var(--card); border-top: 1px solid var(--line);
    height: calc(var(--tabh) + env(safe-area-inset-bottom)); padding-bottom: env(safe-area-inset-bottom); }
  .tab { position: relative; flex: 1; display: flex; flex-direction: column; align-items: center;
    justify-content: center; gap: 2px; font-size: 11px; color: var(--muted); }
  .tab .ic { font-size: 20px; line-height: 1; }
  .tab.on { color: var(--navy); font-weight: 700; background: rgba(13, 59, 102, .07); }
  .tab.on::before { content: ""; position: absolute; top: 0; left: 0; right: 0; height: 3px; background: var(--navy); }

  @media (min-width: 760px) {
    .talk-title { font-size: 15px; }
    .detail-title { font-size: 24px; }
  }
</style>
</head>
<body>
<header class="appbar">
  <div class="brand-main" id="brand"></div>
  <div class="brand-sub" id="brand-sub"></div>
</header>

<main class="content" id="view"></main>

<nav class="tabbar">
  <a class="tab" href="#/program" data-tab="program"><span class="ic">🗓️</span><span>프로그램</span></a>
  <a class="tab" href="#/sessions" data-tab="sessions"><span class="ic">📚</span><span>세션</span></a>
  <a class="tab" href="#/search" data-tab="search"><span class="ic">🔍</span><span>검색</span></a>
  <a class="tab" href="#/my" data-tab="my"><span class="ic">⭐</span><span>내 일정</span></a>
</nav>

<script>
const DATA = __DATA__;

// ───────────────────────── 1. 준비: 데이터 색인 / setup: data indexes ─────────────────────────
function esc(s) {
  return String(s ?? "").replace(/[&<>"']/g, c => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;" }[c]));
}

// id → 객체 를 바로 찾을 수 있게 Map 을 만들어 둔다 (매번 find() 하지 않도록)
// EN: build Maps so we can look up id → object directly (instead of calling find() every time)
const talkById = new Map(DATA.talks.map(t => [t.id, t]));
const abstractById = new Map(DATA.abstracts.map(a => [a.id, a]));
const sessionByCode = new Map(DATA.sessions.map(s => [s.code, s]));
const dayLabel = Object.fromEntries(DATA.talks.map(t => [t.date, t.day_label]));
const rooms = [...new Set(DATA.talks.map(t => t.room))].sort();

// 프로그램에 발표 일정이 없는 초록(포스터 등)도 검색·세션 화면에서 보여 준다
// EN: abstracts without a scheduled talk (posters etc.) are still shown in search and session views
const scheduledAbs = new Set(DATA.talks.map(t => t.abstract_id).filter(Boolean));
const unscheduled = DATA.abstracts.filter(a => !scheduledAbs.has(a.id));

// 검색용 문자열을 미리 소문자로 만들어 둔다 (키를 누를 때마다 만들면 느리다)
// EN: prepare lowercase search strings once (building them on every key press would be slow)
const searchItems = [
  ...DATA.talks.map(t => ({ talk: t, abs: abstractById.get(t.abstract_id) })),
  ...unscheduled.map(a => ({ talk: null, abs: a })),
].map(item => {
  const { talk, abs } = item;
  item.head = [talk?.title, talk?.first_author, abs?.title, ...(abs?.authors || []), ...(abs?.keywords || []),
               talk?.session || abs?.session].join("\n").toLowerCase();
  item.body = (abs?.abstract || "").toLowerCase();
  return item;
});

// ───────────────────────── 2. 북마크 (localStorage) / bookmarks ─────────────────────────
const BM_KEY = "gsk2026.bookmarks";   // step4 와 같은 키 → 실습 중 찍은 북마크가 그대로 이어진다 / same key as step4 → bookmarks carry over
function loadBookmarks() {
  try { return new Set(JSON.parse(localStorage.getItem(BM_KEY) || "[]").map(Number)); }
  catch (e) { return new Set(); }
}
const bookmarks = loadBookmarks();
function saveBookmarks() {
  try { localStorage.setItem(BM_KEY, JSON.stringify([...bookmarks])); return true; }
  catch (e) { return false; }
}
function toggleBookmark(id) {
  if (bookmarks.has(id)) bookmarks.delete(id); else bookmarks.add(id);
  saveBookmarks();
}

// ───────────────────────── 3. 공통 조각 / shared pieces ─────────────────────────
const state = { day: DATA.meta.days[0], room: "전체", q: "", msg: "" };

// 원문을 검색어 기준으로 자른 뒤 조각마다 이스케이프한다
// (이스케이프한 뒤에 찾으면 "amp" 검색이 &amp; 안을 건드린다)
// EN: split the original text by the search term, then escape each piece
// EN: (searching after escaping would make a search for "amp" break &amp;)
function highlight(text, q) {
  if (!q) return esc(text);
  const re = new RegExp("(" + q.replace(/[.*+?^${}()|[\]\\]/g, "\\$&") + ")", "gi");
  return String(text ?? "").split(re).map((part, i) => i % 2 ? `<mark>${esc(part)}</mark>` : esc(part)).join("");
}
// 긴 초록에서 검색어 주변만 잘라 보여 준다
// EN: show only the part of a long abstract around the search term
function snippet(text, q, width = 60) {
  const i = text.toLowerCase().indexOf(q);
  if (i < 0) return "";
  const from = Math.max(0, i - width), to = Math.min(text.length, i + q.length + width);
  return (from > 0 ? "…" : "") + highlight(text.slice(from, to), q) + (to < text.length ? "…" : "");
}

function sessionBadge(code, title) {
  return code ? `<span class="badge" title="${esc(title || sessionByCode.get(code)?.title)}">${esc(code)}</span>` : "";
}

function talkCard(t, { q = "", showDay = false, showRoom = true, clash = false, extra = "" } = {}) {
  const on = bookmarks.has(t.id);
  const plenary = t.kind === "plenary" ? '<span class="badge plenary">기조</span>' : "";
  return `
    <div class="talk ${clash ? "clash" : ""}">
      <div class="talk-time">${showDay ? esc(t.day_label) + "<br>" : ""}${t.time_start}<br>${t.time_end}</div>
      <div class="talk-body">
        <a class="talk-title" href="#/talk/${t.id}">${highlight(t.title, q)}</a>
        <div class="talk-meta">
          ${plenary}${sessionBadge(t.session)}
          <span class="author">${highlight(t.first_author, q)}</span>
          ${showRoom ? `<span class="place">· ${esc(t.room)}</span>` : ""}
          ${clash ? '<span class="clash-badge">⚠ 시간 겹침</span>' : ""}
        </div>
        ${extra}
      </div>
      <button class="bm ${on ? "on" : ""}" data-id="${t.id}" title="북마크">${on ? "★" : "☆"}</button>
    </div>`;
}

function abstractCard(a, { q = "", extra = "" } = {}) {
  return `
    <div class="talk">
      <div class="talk-time"><span class="badge gray">초록</span></div>
      <div class="talk-body">
        <a class="talk-title" href="#/abstract/${a.id}">${highlight(a.title, q)}</a>
        <div class="talk-meta">${sessionBadge(a.session)}<span class="author">${highlight(a.authors[0] || "", q)}</span></div>
        ${extra}
      </div>
    </div>`;
}

function chips(values, current, label = v => v) {
  return values.map(v => `<button class="chip ${v === current ? "on" : ""}" data-v="${esc(v)}">${esc(label(v))}</button>`).join("");
}

// ───────────────────────── 4. 화면(view) 들 / views ─────────────────────────
// 각 view 함수는 HTML 문자열을 돌려준다. route() 가 그것을 #view 에 넣는다.
// EN: each view function returns an HTML string; route() puts it into #view.

function viewProgram() {
  const talks = DATA.talks.filter(t => t.date === state.day && (state.room === "전체" || t.room === state.room));
  return `
    <div class="chips" data-set="day">${chips(DATA.meta.days, state.day, d => dayLabel[d])}</div>
    <div class="chips rooms" data-set="room">${chips(["전체", ...rooms], state.room)}</div>
    <div class="count">${talks.length}건</div>
    ${talks.map(t => talkCard(t, { showRoom: state.room === "전체" })).join("") || '<div class="empty">발표가 없습니다.</div>'}`;
}

function viewSessions() {
  const count = code => DATA.talks.filter(t => t.session === code).length;
  return `
    <h1 class="page-h">세션 ${DATA.sessions.length}개</h1>
    <ul class="slist">${DATA.sessions.map(s => `
      <li><a href="#/session/${encodeURIComponent(s.code)}">
        <span class="badge">${esc(s.code)}</span>
        <span class="stitle">${esc(s.title)}</span>
        <span class="scount">${count(s.code)}편</span></a></li>`).join("")}
    </ul>`;
}

function viewSession(code) {
  const s = sessionByCode.get(code);
  if (!s) return '<div class="empty">세션을 찾을 수 없습니다.</div>';
  const talks = DATA.talks.filter(t => t.session === code);
  const posters = unscheduled.filter(a => a.session === code);
  let html = `<button class="back" data-back>← 뒤로</button>
    <h1 class="page-h"><span class="badge">${esc(code)}</span></h1>
    <div class="session-title">${esc(s.title)}</div>`;
  for (const day of DATA.meta.days) {
    const list = talks.filter(t => t.date === day);
    if (list.length) html += `<h2 class="day-h">${esc(dayLabel[day])}</h2>` + list.map(t => talkCard(t)).join("");
  }
  if (posters.length) {
    html += `<h2 class="day-h">발표 일정 없는 초록 (${posters.length})</h2>` + posters.map(a => abstractCard(a)).join("");
  }
  return html;
}

function viewDetail(talk, abs) {
  if (!talk && !abs) return '<div class="empty">발표를 찾을 수 없습니다.</div>';
  const code = talk?.session || abs?.session;
  const s = sessionByCode.get(code);
  let html = `<button class="back" data-back>← 뒤로</button>`;
  if (talk) {
    const on = bookmarks.has(talk.id);
    html += `<div class="detail-when">
      ${talk.kind === "plenary" ? '<span class="badge plenary">기조강연</span>' : ""}
      ${esc(talk.day_label)} · ${talk.time_start}–${talk.time_end} · ${esc(talk.room)}
      <button class="bm big ${on ? "on" : ""}" data-id="${talk.id}" title="북마크">${on ? "★" : "☆"}</button></div>`;
  } else {
    html += `<div class="detail-when"><span class="badge gray">초록</span> 프로그램에 발표 일정이 없습니다</div>`;
  }
  html += `<h1 class="detail-title">${esc(talk?.title || abs.title)}</h1>`;
  if (s) html += `<a class="detail-session" href="#/session/${encodeURIComponent(code)}">${sessionBadge(code)} ${esc(s.title)}</a>`;
  if (!abs) {
    return html + `<p class="detail-authors">${esc(talk.first_author)}</p><p class="empty">연결된 초록이 없습니다.</p>`;
  }
  html += `<p class="detail-authors">${abs.authors.map(esc).join(", ")}</p>`;
  html += abs.abstract ? `<div class="detail-abstract">${esc(abs.abstract)}</div>` : '<p class="hint">초록 본문이 없습니다.</p>';
  if (abs.keywords.length) {
    html += `<div class="kw">${abs.keywords.map(k => `<button class="kwchip" data-kw="${esc(k)}">${esc(k)}</button>`).join("")}</div>`;
  }
  if (abs.affiliations.length) {
    html += `<details class="affil"><summary>소속 (${abs.affiliations.length})</summary>
      <ol>${abs.affiliations.map(a => `<li>${esc(a)}</li>`).join("")}</ol></details>`;
  }
  return html;
}

function searchResults() {
  const q = state.q.trim().toLowerCase();
  if (q.length < 2) return '<div class="hint">두 글자 이상 입력하세요. 제목·저자·키워드·초록 본문까지 찾습니다.</div>';
  const hits = searchItems.filter(it => it.head.includes(q) || it.body.includes(q));
  const MAX = 100;
  const cards = hits.slice(0, MAX).map(({ talk, abs }) => {
    // 제목·저자에 없고 본문에서만 찾았으면, 본문 일부를 함께 보여 준다
    // EN: if it matched only the abstract body (not title/author), show a snippet of the body
    const inHead = (talk?.title + " " + talk?.first_author + " " + abs?.title).toLowerCase().includes(q);
    const extra = !inHead && abs ? `<div class="snippet">${snippet(abs.abstract, q) || highlight([...abs.authors, ...abs.keywords].join(", "), q)}</div>` : "";
    return talk ? talkCard(talk, { q, showDay: true, extra }) : abstractCard(abs, { q, extra });
  }).join("");
  return `<div class="count">${hits.length}건${hits.length > MAX ? ` (앞 ${MAX}건만 표시)` : ""}</div>${cards || '<div class="empty">결과가 없습니다.</div>'}`;
}

function viewSearch() {
  return `
    <div class="searchbar"><input type="search" id="q" placeholder="제목, 저자, 키워드, 초록 본문" value="${esc(state.q)}" autocomplete="off"></div>
    <div id="results">${searchResults()}</div>`;
}

// 같은 날 북마크를 시작 시각 순으로 훑으며, 앞선 발표가 끝나기 전에 시작하면 겹침
// EN: walk the day's bookmarks in start-time order; a talk that starts before an earlier one ends is a clash
function findClashes(talks) {
  const clash = new Set();
  let latestEnd = "", latestId = null;
  for (const t of talks) {
    if (t.time_start < latestEnd) { clash.add(t.id); clash.add(latestId); }
    if (t.time_end > latestEnd) { latestEnd = t.time_end; latestId = t.id; }
  }
  return clash;
}

function viewMy() {
  const mine = DATA.talks.filter(t => bookmarks.has(t.id));
  let html = `<h1 class="page-h">내 일정 ${mine.length}건</h1>`;
  if (!mine.length) html += '<div class="empty">아직 북마크한 발표가 없습니다. 발표 옆 ☆ 를 눌러 보세요.</div>';
  for (const day of DATA.meta.days) {
    const list = mine.filter(t => t.date === day);
    if (!list.length) continue;
    const clash = findClashes(list);
    html += `<h2 class="day-h">${esc(dayLabel[day])} · ${list.length}건${clash.size ? ` · <span class="clash-badge">겹침 ${clash.size}</span>` : ""}</h2>`;
    html += list.map(t => talkCard(t, { clash: clash.has(t.id) })).join("");
  }
  html += `
    <div class="tools">
      <div class="hint">북마크는 이 브라우저에만 저장됩니다. 다른 PC·브라우저로 옮기려면 파일로 내보내세요.</div>
      <div class="tools-row">
        <button class="btn" id="export">⬇ 내보내기 (JSON)</button>
        <label class="btn">⬆ 불러오기<input type="file" id="import" accept=".json,application/json" hidden></label>
        ${mine.length ? '<button class="btn danger" id="clear">모두 지우기</button>' : ""}
      </div>
      ${state.msg ? `<div class="msg">${esc(state.msg)}</div>` : ""}
    </div>`;
  state.msg = "";
  return html;
}

// ───────────────────────── 5. 라우팅 (location.hash) / routing ─────────────────────────
// 주소 끝의 #/talk/12 같은 부분으로 어떤 화면을 그릴지 정한다.
// hash 를 쓰면 file:// 로 연 HTML 에서도 뒤로가기·새로고침·즐겨찾기가 동작한다.
// EN: the part at the end of the address like #/talk/12 decides which view to draw.
// EN: using the hash, back/refresh/bookmarks work even for an HTML file opened via file://.
const scrollMemo = {};
let currentHash = "";

function route() {
  scrollMemo[currentHash] = window.scrollY;
  currentHash = location.hash || "#/program";
  const [, page, arg] = currentHash.split("/");
  const id = Number(arg);
  const view = {
    program: viewProgram,
    sessions: viewSessions,
    session: () => viewSession(decodeURIComponent(arg || "")),
    search: viewSearch,
    my: viewMy,
    talk: () => { const t = talkById.get(id); return viewDetail(t, abstractById.get(t?.abstract_id)); },
    abstract: () => viewDetail(null, abstractById.get(id)),
  }[page] || viewProgram;
  document.getElementById("view").innerHTML = view();

  const tab = { session: "sessions", talk: null, abstract: null }[page] ?? page;
  document.querySelectorAll(".tab").forEach(a => a.classList.toggle("on", a.dataset.tab === tab));
  window.scrollTo(0, scrollMemo[currentHash] || 0);
  if (page === "search" && !state.q) document.getElementById("q").focus();
}

// 같은 화면을 다시 그린다 (북마크·칩을 바꿨을 때, 스크롤 위치 유지)
// EN: redraw the same view (after a bookmark/chip change), keeping the scroll position
function rerender() {
  const y = window.scrollY;
  scrollMemo[currentHash] = y;
  currentHash = "";       // route() 가 이전 위치를 덮어쓰지 않도록 / so route() doesn't overwrite the previous position
  route();
  window.scrollTo(0, y);
}

// ───────────────────────── 6. 이벤트 / events ─────────────────────────
// #view 안의 내용은 계속 바뀌므로, 클릭은 #view 에 한 번만 걸고 e.target 으로 구분한다 (이벤트 위임)
// EN: the contents of #view keep changing, so attach one click listener to #view and tell clicks apart by e.target (event delegation)
const viewEl = document.getElementById("view");

viewEl.addEventListener("click", e => {
  const bm = e.target.closest(".bm");
  if (bm) { toggleBookmark(Number(bm.dataset.id)); rerender(); return; }

  const chip = e.target.closest(".chip");
  if (chip) { state[chip.parentElement.dataset.set] = chip.dataset.v; rerender(); return; }

  const kw = e.target.closest(".kwchip");
  if (kw) { state.q = kw.dataset.kw; location.hash = "#/search"; return; }

  if (e.target.closest("[data-back]")) {
    if (history.length > 1) history.back(); else location.hash = "#/program";
    return;
  }
  if (e.target.id === "export") exportBookmarks();
  if (e.target.id === "clear" && confirm("북마크를 모두 지울까요?")) {
    bookmarks.clear(); saveBookmarks(); rerender();
  }
});

// 검색창: 입력할 때마다 결과 부분만 다시 그린다 (입력창을 다시 만들면 커서가 사라진다)
// EN: search box: on each input redraw only the results (re-creating the input box would lose the cursor)
viewEl.addEventListener("input", e => {
  if (e.target.id !== "q") return;
  state.q = e.target.value;
  document.getElementById("results").innerHTML = searchResults();
});

viewEl.addEventListener("change", e => {
  if (e.target.id === "import" && e.target.files[0]) importBookmarks(e.target.files[0]);
});

function exportBookmarks() {
  const payload = { app: DATA.meta.name, exported: new Date().toISOString(), bookmarks: [...bookmarks] };
  const blob = new Blob([JSON.stringify(payload, null, 1)], { type: "application/json" });
  const a = document.createElement("a");
  a.href = URL.createObjectURL(blob);
  a.download = "bookmarks.json";
  document.body.appendChild(a);
  a.click();
  a.remove();
  setTimeout(() => URL.revokeObjectURL(a.href), 1000);
}

function importBookmarks(file) {
  const reader = new FileReader();
  reader.onload = () => {
    try {
      const data = JSON.parse(reader.result);
      const ids = (Array.isArray(data) ? data : data.bookmarks).map(Number).filter(id => talkById.has(id));
      const before = bookmarks.size;
      ids.forEach(id => bookmarks.add(id));
      state.msg = `${ids.length}건 중 ${bookmarks.size - before}건을 새로 추가했습니다.` + (saveBookmarks() ? "" : " (브라우저 저장이 막혀 있어 이번 화면에서만 유지됩니다)");
    } catch (err) {
      state.msg = "불러오기 실패: 올바른 bookmarks.json 파일이 아닙니다.";
    }
    rerender();
  };
  reader.readAsText(file);
}

window.addEventListener("hashchange", route);

// ───────────────────────── 7. 시작 / start ─────────────────────────
document.title = `${DATA.meta.name} 프로그램`;
document.getElementById("brand").textContent = `${DATA.meta.name} 프로그램`;
document.getElementById("brand-sub").textContent =
  `${DATA.meta.place} · ${dayLabel[DATA.meta.days[0]]} – ${dayLabel[DATA.meta.days.at(-1)]}`;
// 학회 기간 중이면 오늘 날짜를 기본으로 보여 준다
// EN: during the conference, start on today's date
const today = new Date().toISOString().slice(0, 10);
if (DATA.meta.days.includes(today)) state.day = today;
route();
</script>
</body>
</html>

In [ ]:
build("app.html", "out_step5.html")
preview("out_step5.html")

In [ ]:
download("out_step5.html")

## 과제 아이디어 / Assignment ideas

난이도 순서입니다. Step 5 템플릿 셀(`app.html`)을 고친 뒤 build 셀을 다시 실행해 확인하세요.

1. 첫 화면 위쪽에 "총 발표 N건 · 세션 M개" 요약 표시
2. 발표 카드에 세션 제목을 마우스를 올렸을 때 보이게(`title` 속성)
3. 내 일정에서 북마크 개수를 탭 이름 옆에 배지로 표시
4. 다크 모드 버튼 (CSS 변수 바꾸기)
5. 내 일정을 **인쇄용**으로 깔끔하게 (`@media print`)
6. 키워드 구름: 초록 키워드를 많이 나온 순서로 모아 보여 주고, 누르면 검색
7. 내 일정을 캘린더 파일(`.ics`)로 내보내기

파이썬 쪽 과제: 세션 하나만 담은 작은 HTML 만들기 (`DATA` 를 걸러서 `build()`).
막히면 완성본 코드에서 비슷한 부분을 찾아보세요.

> ⚠️ 만든 HTML 에는 초록 본문까지 데이터 전체가 들어 있습니다. 공개된 곳(웹·GitHub·SNS)에 올리지 마세요.

---

🇬🇧 **English**

In order of difficulty. Edit the Step 5 template cell (`app.html`), then re-run the build cell to check.

1. Show a summary like "N talks · M sessions" at the top of the first screen
2. Show the session title when hovering over a talk card (`title` attribute)
3. Show the bookmark count as a badge next to the My Schedule tab name
4. A dark-mode button (switch CSS variables)
5. A clean **printable** My Schedule (`@media print`)
6. Keyword cloud: collect abstract keywords by frequency; click one to search
7. Export My Schedule as a calendar file (`.ics`)

Python-side assignment: make a small HTML with just one session (filter `DATA`, then `build()`).
If you get stuck, look for similar parts in the finished app's code.

> ⚠️ The HTML you made contains the whole dataset, including abstract text. Don't post it anywhere public (web, GitHub, social media).